<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/local_tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/jarwo_tts")

RAW_DIR = ROOT / "raw_audio"
PREP_DIR = ROOT / "prepared"
EXPORT_DIR = ROOT / "export"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PREP_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT       :", ROOT)
print("RAW_DIR    :", RAW_DIR)
print("PREP_DIR   :", PREP_DIR)
print("EXPORT_DIR :", EXPORT_DIR)

In [ ]:
# 2 — Install dependency + Piper
!apt-get update -qq
!apt-get install -y -qq \
    build-essential \
    cmake \
    ninja-build \
    ffmpeg \
    git \
    espeak-ng

!pip install -q --upgrade pip setuptools wheel
!pip install -q huggingface_hub soundfile

In [ ]:
%cd /content

!rm -rf piper1-gpl
!git clone --depth 1 https://github.com/OHF-Voice/piper1-gpl.git

%cd /content/piper1-gpl

!pip install -q -e ".[train]"

!chmod +x build_monotonic_align.sh
!./build_monotonic_align.sh

!python3 setup.py build_ext --inplace

In [ ]:
# 3 — Pastikan GPU aktif

import torch

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print(
        "VRAM    :",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    raise RuntimeError(
        "GPU tidak aktif. Colab → Runtime → Change runtime type → T4 GPU"
    )

In [ ]:
# 4 — Buat 30 kalimat latihan
from pathlib import Path

PROMPTS = [
    "Halo, selamat pagi.",
    "Ada yang bisa saya bantu?",
    "Sebentar, saya periksa dulu.",
    "Sekarang jam tujuh lewat lima menit.",
    "Baterai perangkat masih delapan puluh persen.",
    "Penggunaan prosesor sekarang cukup rendah.",
    "Memori yang tersedia sekitar dua gigabyte.",
    "Suhu perangkat sekarang empat puluh lima derajat.",
    "Koneksi Wi-Fi sedang aktif.",
    "Perangkat sudah terhubung ke jaringan.",
    "Sepertinya semuanya berjalan dengan normal.",
    "Baik, saya akan mencoba memeriksanya.",
    "Ada sedikit masalah yang perlu diperiksa.",
    "Saya belum menemukan jawabannya.",
    "Tidak masalah, kita bisa mencoba lagi.",
    "Apa yang ingin kamu lakukan sekarang?",
    "Saya rasa itu ide yang cukup bagus.",
    "Kalau begitu kita gunakan cara yang lain.",
    "Tunggu sebentar, saya sedang memprosesnya.",
    "Sudah selesai, semuanya berjalan dengan baik.",
    "Hari ini terasa cukup tenang.",
    "Saya sedang menunggu perintah berikutnya.",
    "Penyimpanan masih memiliki ruang yang cukup.",
    "Alamat IP perangkat sudah ditemukan.",
    "Layanan utama sedang berjalan.",
    "Apakah kamu ingin saya memeriksanya lagi?",
    "Oke, saya mengerti.",
    "Hmm, sepertinya ada sesuatu yang berbeda.",
    "Baiklah, kita lanjutkan.",
    "Sampai nanti, jangan lupa istirahat."
]

PROMPT_FILE = ROOT / "prompts.txt"

with open(PROMPT_FILE, "w", encoding="utf-8") as f:
    for i, text in enumerate(PROMPTS, 1):
        f.write(f"{i:04d}|{text}\n")

print(f"Created: {PROMPT_FILE}")
print()
print(open(PROMPT_FILE, encoding="utf-8").read())

In [ ]:
# MyDrive/jarwo_tts/raw_audio/
# 5 — Rekam suara target

In [ ]:
# 6 Cek apakah 30 audio sudah ada
from pathlib import Path

missing = []

for i in range(1, len(PROMPTS) + 1):
    stem = f"{i:04d}"

    found = [
        p for p in RAW_DIR.iterdir()
        if p.is_file() and p.stem == stem
    ]

    if len(found) == 0:
        missing.append(stem)

if missing:
    print("Audio yang belum ada:")
    print(missing)
else:
    print("PASS semua audio ditemukan ✅")

In [ ]:
# 7 — Convert semuanya menjadi WAV 22.05 kHz

import subprocess
import shutil
from pathlib import Path

WAV_DIR = PREP_DIR / "wavs"

if WAV_DIR.exists():
    shutil.rmtree(WAV_DIR)

WAV_DIR.mkdir(parents=True, exist_ok=True)

metadata = []

for i, text in enumerate(PROMPTS, 1):

    stem = f"{i:04d}"

    candidates = [
        p for p in RAW_DIR.iterdir()
        if p.is_file() and p.stem == stem
    ]

    if len(candidates) == 0:
        raise FileNotFoundError(f"Tidak ditemukan audio untuk {stem}")

    if len(candidates) > 1:
        raise RuntimeError(
            f"Ada lebih dari satu audio dengan nomor {stem}: {candidates}"
        )

    src = candidates[0]
    dst = WAV_DIR / f"{stem}.wav"

    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-loglevel", "error",
            "-i", str(src),
            "-vn",
            "-ac", "1",
            "-ar", "22050",
            "-sample_fmt", "s16",
            str(dst)
        ],
        check=True
    )

    if "|" in text:
        raise ValueError(f"Karakter | tidak boleh ada dalam teks: {text}")

    metadata.append(f"{stem}.wav|{text}")


METADATA_FILE = PREP_DIR / "metadata.csv"

with open(METADATA_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(metadata))

print("WAV DIR :", WAV_DIR)
print("Metadata:", METADATA_FILE)
print("Samples :", len(metadata))

In [ ]:
# 8 Check Quality

import soundfile as sf
import numpy as np

durations = []

problems = []

for wav in sorted(WAV_DIR.glob("*.wav")):

    info = sf.info(wav)

    duration = info.frames / info.samplerate

    durations.append(duration)

    if info.samplerate != 22050:
        problems.append(f"{wav.name}: sample rate {info.samplerate}")

    if info.channels != 1:
        problems.append(f"{wav.name}: channels {info.channels}")

    if duration < 0.5:
        problems.append(f"{wav.name}: terlalu pendek ({duration:.2f}s)")

    if duration > 12:
        problems.append(f"{wav.name}: terlalu panjang ({duration:.2f}s)")


print("Jumlah audio :", len(durations))
print("Total menit  :", round(sum(durations) / 60, 2))
print("Rata-rata    :", round(np.mean(durations), 2), "detik")
print("Terpendek    :", round(min(durations), 2), "detik")
print("Terpanjang   :", round(max(durations), 2), "detik")

print()

if problems:
    print("WARNING:")
    for p in problems:
        print(" -", p)
else:
    print("Dataset check PASS ✅")

In [ ]:
# 9.Download preatrained

from huggingface_hub import hf_hub_download
from pathlib import Path

BASE_DIR = Path("/content/base_model")
BASE_DIR.mkdir(exist_ok=True)

BASE_CKPT = hf_hub_download(
    repo_id="rhasspy/piper-checkpoints",
    repo_type="dataset",
    filename=(
        "id/id_ID/news_tts/medium/"
        "epoch=4927-step=232092.ckpt"
    ),
    local_dir=str(BASE_DIR)
)

print("Checkpoint:")
print(BASE_CKPT)

In [ ]:
# 10 — Copy dataset dari Drive ke storage Colab

import shutil
from pathlib import Path

LOCAL_DATA = Path("/content/jarwo_dataset")

if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)

LOCAL_WAV = LOCAL_DATA / "wavs"
LOCAL_WAV.mkdir(parents=True)

for wav in WAV_DIR.glob("*.wav"):
    shutil.copy2(wav, LOCAL_WAV / wav.name)

LOCAL_METADATA = LOCAL_DATA / "metadata.csv"
shutil.copy2(METADATA_FILE, LOCAL_METADATA)

LOCAL_CONFIG = LOCAL_DATA / "config.json"

print("Audio :", len(list(LOCAL_WAV.glob('*.wav'))))
print("CSV   :", LOCAL_METADATA)

In [ ]:
# 11 Fine Tune Pertama

import subprocess
import shutil
from pathlib import Path

RUN_DIR = Path("/content/jarwo_runs")

if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)

RUN_DIR.mkdir(parents=True)

cmd = [
    "python3",
    "-m",
    "piper.train",
    "fit",

    "--data.voice_name",
    "jarwo_custom",

    "--data.csv_path",
    str(LOCAL_METADATA),

    "--data.audio_dir",
    str(LOCAL_WAV),

    "--model.sample_rate",
    "22050",

    "--data.espeak_voice",
    "id",

    "--data.cache_dir",
    "/content/jarwo_cache",

    "--data.config_path",
    str(LOCAL_CONFIG),

    "--data.batch_size",
    "4",

    "--data.validation_split",
    "0.10",

    "--data.num_test_examples",
    "2",

    "--data.num_workers",
    "2",

    "--model.warmstart_ckpt",
    str(BASE_CKPT),

    "--model.mos_metric",
    "none",

    "--trainer.accelerator",
    "gpu",

    "--trainer.devices",
    "1",

    "--trainer.default_root_dir",
    str(RUN_DIR),

    "--trainer.max_steps",
    "1500",

    "--trainer.max_epochs",
    "500",

    "--trainer.check_val_every_n_epoch",
    "10",

    "--trainer.log_every_n_steps",
    "10",
]

print("Starting Piper fine-tuning...")

subprocess.run(
    cmd,
    cwd="/content/piper1-gpl",
    check=True
)

In [ ]:
# 12 — Cari checkpoint hasil training
from pathlib import Path

ckpts = list(RUN_DIR.rglob("*.ckpt"))

print("Checkpoint ditemukan:", len(ckpts))

for ckpt in ckpts:
    print(
        ckpt,
        round(ckpt.stat().st_size / 1024**2, 1),
        "MB"
    )

last_candidates = [
    p for p in ckpts
    if p.name == "last.ckpt"
]

if last_candidates:
    FINAL_CKPT = max(
        last_candidates,
        key=lambda p: p.stat().st_mtime
    )
else:
    FINAL_CKPT = max(
        ckpts,
        key=lambda p: p.stat().st_mtime
    )

print("Dipilih:")
print(FINAL_CKPT)

In [ ]:
# 14 Backup Google

import shutil

DRIVE_CKPT = ROOT / "jarwo_learning_last.ckpt"

shutil.copy2(
    FINAL_CKPT,
    DRIVE_CKPT
)

print("Saved:")
print(DRIVE_CKPT)

In [ ]:
# 15 Export

import subprocess
from pathlib import Path

ONNX_FILE = EXPORT_DIR / "id_ID-jarwo-medium.onnx"

subprocess.run(
    [
        "python3",
        "-m",
        "piper.train.export_onnx",

        "--checkpoint",
        str(FINAL_CKPT),

        "--output-file",
        str(ONNX_FILE),
    ],
    cwd="/content/piper1-gpl",
    check=True
)

print("ONNX created:")
print(ONNX_FILE)

In [ ]:
# 15 — Copy config ONNX
import shutil
from pathlib import Path

ONNX_JSON = Path(
    str(ONNX_FILE) + ".json"
)

shutil.copy2(
    LOCAL_CONFIG,
    ONNX_JSON
)

print("Model :", ONNX_FILE)
print("Config:", ONNX_JSON)

In [ ]:
# 16 Custom Voice Test
import subprocess

TEST_WAV = EXPORT_DIR / "test_jarwo.wav"

TEST_TEXT = (
    "Halo. Sekarang saya sudah berjalan secara lokal. "
    "Ada yang bisa saya bantu?"
)

subprocess.run(
    [
        "python3",
        "-m",
        "piper",

        "-m",
        str(ONNX_FILE),

        "-f",
        str(TEST_WAV),

        "--",
        TEST_TEXT,
    ],
    cwd="/content/piper1-gpl",
    check=True
)

print(TEST_WAV)

In [ ]:
# 17 Testing
from IPython.display import Audio, display

display(
    Audio(
        str(TEST_WAV),
        autoplay=False
    )
)